# Conciliación de facturas de transportistas

Este cuaderno muestra, paso a paso, qué hace la herramienta: lee las facturas de los transportistas, las compara con el registro de cargas acordadas y lista lo que una persona debe revisar, con la razón.

**Cómo usarlo:** en el menú elija `Runtime > Run all` (en español: Entorno de ejecución > Ejecutar todo) y espere. Tarda entre 1 y 2 minutos. No necesita ninguna clave ni escribir nada.

La herramienta no aprueba ni paga facturas: solo avisa qué conviene revisar.

## Paso 1. Descargar el proyecto

Se descarga la versión entregada del proyecto (etiqueta `entrega-v1`) y se instalan los programas de apoyo. Pueden aparecer varias líneas de texto o avisos; no afectan el resultado.

In [ ]:
!rm -rf freight-invoice-recon
!git clone --depth 1 --branch entrega-v1 https://github.com/yarecast/freight-invoice-recon.git
%cd freight-invoice-recon
!pip install -q -r requirements.txt

## Paso 2. Ejecutar la herramienta

Lee las 18 facturas, las compara con el registro de cargas y genera el reporte. Este modo no usa internet ni claves: usa las lecturas que ya se guardaron en la carpeta `extractions/`.

In [ ]:
!python run.py

## Paso 3. El reporte

Esta es la lista de lo que una persona debe revisar. Empiece por la tabla "Excepciones por severidad": cada fila trae la razón y la acción sugerida.

In [ ]:
from IPython.display import HTML, display

with open("output/reporte.html", encoding="utf-8") as fh:
    display(HTML(fh.read()))

## Paso 4. Una factura y lo que la herramienta leyó

A la izquierda, la factura 17, que tiene notas escritas a mano. A la derecha, los datos que el modelo leyó de ella, cada uno con el texto de la factura que lo respalda. El reporte marca esta factura para revisión porque el total escrito a mano no coincide con lo pactado.

In [ ]:
import base64, html, json
from pathlib import Path

import pymupdf
from IPython.display import HTML, display

pdf_path = next(Path("data/facturas").glob("factura_17*.pdf"))
json_path = Path("extractions") / (pdf_path.stem + ".json")

with pymupdf.open(pdf_path) as doc:
    png = doc[0].get_pixmap(dpi=110).tobytes("png")
image = base64.b64encode(png).decode("ascii")

data = json.loads(json_path.read_text(encoding="utf-8"))
pretty = html.escape(json.dumps(data, indent=2, ensure_ascii=False))

display(HTML(
    "<table><tr>"
    f'<td style="vertical-align:top;width:50%"><img src="data:image/png;base64,{image}" style="max-width:100%"></td>'
    f'<td style="vertical-align:top"><pre style="font-size:12px;white-space:pre-wrap">{pretty}</pre></td>'
    "</tr></table>"
))

## Opcional. Ejecutar con el modelo en vivo

No es necesario para ver el resultado. Esta celda está comentada. Para usarla, guarde su propia clave en `Secrets` (icono de llave a la izquierda) con los nombres `GEMINI_API_KEY` y `GEMINI_MODEL`, quite el `#` de cada línea y ejecútela. Nunca escriba la clave dentro del cuaderno.

In [ ]:
# import os
# from google.colab import userdata
#
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
# os.environ["GEMINI_MODEL"] = userdata.get("GEMINI_MODEL")
# !python run.py --live --only-missing